### CV와 LB의 차이가 커서, 예측 분포를 확인하는 과정을 포함시켰다.
예측 분포를 보는 것은 모델 출력 점검이라서 해커톤 규칙에 어긋나지 않음

#### 0) 준비: 03_lr_l0과 같은 조건으로 데이터 로드 + LR L0 CV 재실행
아래 셀(OOF 비교표)이 `train`, `le`, `result`를 쓰므로, 이 노트북만 위에서부터 실행해도 되도록 준비 셀을 둔다. (약 20초)

In [1]:
import sys; sys.path.append('..')

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold

from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import to_binary
from src.validation import seed_everything, encode_target, run_cv, save_submission

seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
X = to_binary(train, genes)
X_test = to_binary(test, genes)

make_lr = lambda cw='balanced', C=1.0: (
    lambda: LogisticRegression(max_iter=3000, class_weight=cw, C=C, random_state=SEED)
)
result = run_cv(make_lr(), X, y, X_test)   # 03_lr_l0와 동일: CV 0.3432

[fold 0] Macro F1 = 0.3455
[fold 1] Macro F1 = 0.3289
[fold 2] Macro F1 = 0.3527
[fold 3] Macro F1 = 0.3419
[fold 4] Macro F1 = 0.3470
CV Macro F1 = 0.3432 ± 0.0079 (OOF 0.3445)


In [2]:
import pandas as pd

s = pd.read_csv('../submissions/1109_eunji_lr_l0_0.343.csv')
sample = pd.read_csv('../data/sample_submission.csv')

print('행·열:', s.shape)                                  # (2546, 2)
print('ID 순서 일치:', (s.ID == sample.ID).all())          # True
print('예측된 클래스 수:', s.SUBCLASS.nunique())           # 26 근처
print('결측:', s.SUBCLASS.isna().sum())                   # 0
print(s.SUBCLASS.value_counts(normalize=True).round(3).head(10))

행·열: (2546, 2)
ID 순서 일치: True
예측된 클래스 수: 26
결측: 0
SUBCLASS
COAD      0.188
STES      0.098
BRCA      0.093
KIPAN     0.072
THYM      0.055
OV        0.050
GBMLGG    0.041
KIRC      0.036
PCPG      0.033
HNSC      0.033
Name: proportion, dtype: float64


#### ➡️ 형식은 전부 정상이나, 예측분포가 한쪽으로 쏠려 있다.
- COAD가 train보다 5배 많이 예측됐다.(01_eda_data_target.ipynb에 의하면 COAD는 3.6%로 test 예측 비율 18.8%에 비해 약 5배 차이)
- test의 약 480명이 COAD로 예측 된 것인데, train 비율대로라면 90명 정도가 자연스러움. 
- 틀린 COAD 예측이 많으면 COAD 정밀도가 떨어지고, 그 환자들의 원래 클래스 재현율도 함께 떨어져서 Macro F1이 크게 깎인다. 
-> CV와 LB차이의 주요 원인일 가능성

In [3]:
oof_pred = pd.Series(le.inverse_transform(result['oof_proba'].argmax(axis=1)))
compare = pd.DataFrame({
    'train 실제': train['SUBCLASS'].value_counts(normalize=True),
    'train OOF 예측': oof_pred.value_counts(normalize=True),
    'test 예측': s.SUBCLASS.value_counts(normalize=True),
}).round(3).sort_values('test 예측', ascending=False)
compare.head(12)

,train 실제,train OOF 예측,test 예측
COAD,0.036,0.039,0.188
STES,0.061,0.048,0.098
BRCA,0.127,0.106,0.093
KIPAN,0.083,0.073,0.072
THYM,0.016,0.078,0.055
OV,0.041,0.065,0.050
GBMLGG,0.074,0.060,0.041
KIRC,0.054,0.060,0.036
PCPG,0.024,0.039,0.033
PRAD,0.043,0.054,0.033


---
# CV(0.3432)와 LB(0.2533) 차이 원인 분석

> **원칙**: 원인 분석과 해결책 선택은 **train(OOF)만** 사용한다. test는 위의 예측 분포 확인(모델 출력 점검)까지만 보고, test 분포를 더 분석하거나 그에 맞춰 모델을 조정하지 않는다 (해커톤 규칙: test EDA 인사이트 반영 = Data Leakage).

| 원인 후보 | 확인 방법 | 섹션 |
| :--- | :--- | :-: |
| A. CV 설계가 낙관적 (동일 샘플이 train/val fold에 나뉨) | 동일 프로파일을 같은 fold로 묶은 그룹 CV와 비교 | 1 |
| B. 겹치는 라벨 (KIPAN↔KIRC, GBMLGG↔LGG) | 동일 프로파일·다른 라벨 쌍, 클래스별 F1, 혼동 행렬 | 2 |
| C. 변이가 없는/적은 샘플의 쏠림 (THYM 과다 예측) | 변이 0개 행의 OOF 예측 | 3 |
| D. train과 test의 입력 분포 차이 (COAD 쏠림) | train OOF vs test 예측 비율 (위 표) | 4 |

## 1) 원인 후보 A: CV 설계가 낙관적인가?
유전자 프로파일이 완전히 같은 샘플끼리 같은 fold에 들어가도록 `StratifiedGroupKFold`로 다시 검증한다. 점수가 크게 떨어지면 기존 CV가 부풀려졌다는 뜻이다.

In [4]:
profile_key = pd.util.hash_pandas_object(train[genes], index=False)
groups = pd.factorize(profile_key)[0]
folds_group = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(X, y, groups))

r_group = run_cv(make_lr(), X, y, folds=folds_group, verbose=False)
print(f"StratifiedKFold (공통 CV) : {result['cv_mean']:.4f} ± {result['cv_std']:.4f}")
print(f"StratifiedGroupKFold     : {r_group['cv_mean']:.4f} ± {r_group['cv_std']:.4f}")

StratifiedKFold (공통 CV) : 0.3432 ± 0.0079
StratifiedGroupKFold     : 0.3520 ± 0.0104


#### ➡️ 결과: 원인 아님
- 그룹 CV **0.3520** ≥ 공통 CV 0.3432 → 중복 샘플 때문에 CV가 부풀려진 것이 **아니다**.
- (동일 프로파일이 서로 다른 라벨을 갖기 때문에, 오히려 다른 fold에 나뉘면 점수가 깎인다.)
- 공통 CV(Stratified 5-Fold)는 그대로 유지해도 된다.

## 2) 원인 후보 B: 겹치는 라벨

In [5]:
dup = train.assign(key=profile_key, n_mut=X.sum(axis=1))
dup = dup[dup.key.duplicated(keep=False)]
pair_labels = dup.groupby('key')['SUBCLASS'].apply(lambda s: ' / '.join(sorted(set(s))))
print('동일 프로파일 그룹 수:', dup.key.nunique(), '| 포함된 행 수:', len(dup))
pair_labels.value_counts().head(5)

동일 프로파일 그룹 수: 451 | 포함된 행 수: 1016


SUBCLASS
KIPAN / KIRC                                                                                                272
GBMLGG / LGG                                                                                                173
LAML                                                                                                          3
BRCA / CESC / GBMLGG / KIPAN / KIRC / LAML / LGG / LUSC / PAAD / PCPG / PRAD / SARC / STES / THCA / THYM      1
LGG / THCA                                                                                                    1
Name: count, dtype: int64

In [6]:
oof_pred = result['oof_proba'].argmax(axis=1)
f1_by_class = pd.Series(f1_score(y, oof_pred, average=None), index=le.classes_).round(3)
print('클래스별 F1 (낮은 순):')
print(f1_by_class.sort_values().head(10).to_string())

overlap = ['KIPAN', 'KIRC', 'GBMLGG', 'LGG']
cm = pd.DataFrame(confusion_matrix(y, oof_pred), index=le.classes_, columns=le.classes_)
cm.loc[overlap, overlap]

클래스별 F1 (낮은 순):
CESC     0.091
KIRC     0.138
KIPAN    0.147
SARC     0.187
BLCA     0.188
THYM     0.189
PAAD     0.208
LIHC     0.215
LUAD     0.231
PCPG     0.235


,KIPAN,KIRC,GBMLGG,LGG
KIPAN,71,274,3,0
KIRC,229,49,4,1
GBMLGG,10,3,117,196
LGG,1,0,140,68


#### ➡️ 결과: 점수 상한을 낮추는 구조적 문제 (CV·LB 공통)
- 프로파일이 같은데 라벨이 다른 쌍: **KIPAN / KIRC 272쌍, GBMLGG / LGG 173쌍**
  - KIRC 334명 중 272명, LGG 229명 중 173명이 통합 코호트에 **같은 환자로 이중 포함**된 것으로 보인다.
  - 입력이 같은데 정답이 둘 → **어떤 모델도 둘 다 맞힐 수 없음**
- 클래스별 F1: **KIRC 0.138, KIPAN 0.147**, GBMLGG 0.281, LGG 0.265 (CESC 0.091 다음으로 최하위권)
- 혼동 행렬: **KIPAN 실제 348명 중 274명을 KIRC로, KIRC 283명 중 229명을 KIPAN으로** 예측 → 서로 **맞바꿔** 예측하는 패턴
  - 같은 프로파일의 두 사본이 서로 다른 fold에 들어가면, 검증 사본은 학습 사본의 '다른 라벨'로 예측되기 때문 (GBMLGG↔LGG도 동일: 196 / 140)
- CV와 LB 모두에 똑같이 작용하므로 **"차이"의 원인이라기보다 전체 점수의 상한**을 낮추는 요인이다.

## 3) 원인 후보 C: 변이가 없는/적은 샘플

In [7]:
n_mut = X.sum(axis=1)
oof_label = pd.Series(le.classes_[oof_pred], index=train.index)
zero = n_mut == 0
print('변이 0개 행:', zero.sum(), '개')
print('  실제 라벨:', train.loc[zero, 'SUBCLASS'].value_counts().head(5).to_dict())
print('  OOF 예측  :', oof_label[zero].value_counts().head(5).to_dict())

변이 0개 행: 94 개
  실제 라벨: {'THYM': 30, 'THCA': 18, 'LAML': 12, 'BRCA': 5, 'KIPAN': 4}
  OOF 예측  : {'THYM': 94}


#### ➡️ 결과: THYM 과다 예측의 직접 원인 (train에서도 발생 → CV로 개선 가능)
- 변이 0개 행 **94개가 전부 THYM으로 예측**된다 (실제는 THYM 30, THCA 18, LAML 12 …).
- 입력이 모두 0이라 모델은 절편(클래스 기본값)만으로 판단 → `class_weight='balanced'`로 가중치가 큰 소수 클래스 THYM이 선택된다.
- 그래서 OOF에서도 THYM이 실제 1.6% → 예측 7.8%로 과다 예측된다.

## 4) 원인 후보 D: train과 test의 입력 분포 차이
위 비교표 요약:

| 클래스 | train 실제 | train OOF 예측 | test 예측 |
| :--- | :-: | :-: | :-: |
| COAD | 3.6% | **3.9%** | **18.8%** |
| STES | 6.1% | 4.8% | 9.8% |

#### ➡️ 결과: CV-LB 차이의 **주요 원인으로 추정**
- train 안(OOF)에서는 COAD를 실제 비율대로 예측하는데, **test에서만 5배**로 늘어난다 → 모델 문제가 아니라 **test 입력이 train과 다르게 생겼기 때문**으로 보인다 (test에만 빈칸이 있는 것도 생성 과정이 다를 수 있다는 정황).
- 데이터는 주어진 것이므로 바꿀 수 없고, test에 맞춰 조정하는 것은 규칙 위반 + LB 과적합이다.
- **할 수 있는 대응**: test를 보지 않고, 입력 분포 변화에 **일반적으로 덜 민감한 방법**을 **CV 기준으로** 고른다 (규제 강화, 행 단위 정규화 등). LB는 방향 확인용으로만 쓴다.

## 5) 원인 정리

| 원인 | 판단 | CV로 개선 가능? | 대응 |
| :--- | :-: | :-: | :--- |
| A. CV 설계 (중복 샘플) | ❌ 원인 아님 (그룹 CV 0.352) | — | 공통 CV 유지 |
| B. 겹치는 라벨 (KIPAN↔KIRC, GBMLGG↔LGG) | ⚠️ 점수 상한 요인 | 일부 | 두 쌍의 확률 균형 조정 (OOF 기준) |
| C. 변이 0개 → 전부 THYM | ⚠️ 과다 예측 원인 | ✅ | 규제 강화, 가중치 조정 |
| D. test 입력 분포 차이 (COAD 5배) | 🔴 **주요 원인 추정** | 간접적으로 | 분포 변화에 강한 방법을 CV로 선택 |

---
# 해결 방법 실험 (모두 CV 기준, test 미사용)

In [8]:
exp = {}

# (1) class_weight: balanced vs None
exp['cw=None, C=1'] = run_cv(make_lr(cw=None), X, y, verbose=False)['cv_mean']
exp['cw=balanced, C=1 (기준)'] = result['cv_mean']

# (2) 규제 강도 C (작을수록 강한 규제)
for C in [0.03, 0.1, 0.3]:
    exp[f'cw=balanced, C={C}'] = run_cv(make_lr(C=C), X, y, verbose=False)['cv_mean']

# (3) 행 단위 정규화: 변이가 많은 샘플일수록 각 변이의 크기를 줄임 (행 내부 연산 → 누수 아님)
X_rownorm = X.div(np.sqrt(n_mut.clip(lower=1)), axis=0)
exp['row-norm, C=1'] = run_cv(make_lr(), X_rownorm, y, verbose=False)['cv_mean']

pd.Series(exp, name='CV Macro F1').round(4).sort_values(ascending=False).to_frame()

,CV Macro F1
"cw=balanced, C=0.1",0.3818
"cw=balanced, C=0.03",0.3739
"cw=balanced, C=0.3",0.3664
"row-norm, C=1",0.3645
"cw=balanced, C=1 (기준)",0.3432
"cw=None, C=1",0.2997


#### ➡️ 실험 결과 요약 (기준 0.3432)

| 설정 | CV Macro F1 | 변화 | 비고 |
| :--- | :-: | :-: | :--- |
| **balanced, C=0.1** | **0.3818** | **+0.039** | 최고. 규제를 강하게 하면 희소 피처 과적합 감소 |
| balanced, C=0.03 | 0.3739 | +0.031 | |
| balanced, C=0.3 | 0.3664 | +0.023 | |
| row-norm, C=1 | 0.3645 | +0.021 | 변이 개수 차이에 덜 민감 |
| balanced, C=1 (기준) | 0.3432 | — | |
| class_weight=None | 0.2997 | -0.044 | THYM 쏠림은 사라지지만 소수 클래스 재현율 급락 → `balanced` 유지 |
| (참고) 침묵 변이 제외 | 0.3242 | -0.019 | 침묵 변이도 유용한 정보 → 제외하지 않음 |

- **규제 강화(C=0.1)가 가장 효과적**: 희소한 변이 하나하나에 과하게 반응하지 않게 되어 일반화에 유리하다. 입력 분포가 달라지는 test에서도 도움이 될 가능성이 크다.
- **행 단위 정규화**도 개선되며, "변이가 많은 샘플"이 특정 클래스로 쏠리는 현상을 줄이는 방향이라 test 분포 차이 대응 후보로 계속 검토한다.
- 단, C=0.1에서도 THYM OOF 예측 비율은 약 10%로 여전히 높다 → 변이 0개 행 처리는 추가 과제.

## 다음 단계
1. **제출 후보: LR L0, `balanced`, C=0.1** (CV 0.3818) → 제출해서 LB가 CV 개선 방향을 따라오는지 확인
   - ⚠️ 이 모델도 test에서 COAD 예측 비율이 **22.5%**로 오히려 늘었다 (아래 셀 출력). CV는 올랐지만 **원인 D(test 분포 차이)는 해결되지 않았으므로 LB 개선 폭은 CV보다 작을 수 있다.**
   - 이 비율을 보고 COAD 예측을 손으로 줄이는 것은 금지 (test 정보 활용 + LB 과적합)
   - ✅ 제출 결과: LB 0.2739 (C=1 대비 +0.021) → CV 개선 방향이 LB에도 반영됨. 단 CV−LB 차이는 0.108로 유지
2. C를 0.05~0.2 범위에서 더 세밀하게 탐색 + 행 정규화와 조합
3. 변이 0개 행(THYM 쏠림) 대응: 변이 수 요약 피처 추가 등 → CV로 판단
4. 겹치는 쌍(KIPAN↔KIRC, GBMLGG↔LGG): OOF 확률로 두 클래스 간 배율을 조정해 Macro F1 최대화 (train OOF로만 튜닝)
5. 변이 유형 피처(Missense/Nonsense/Frameshift) 추가 후 LightGBM 재비교

In [9]:
# 제출 후보 생성: LR L0, balanced, C=0.1 (submissions/는 git 제외)
best = run_cv(make_lr(C=0.1), X, y, X_test)
save_submission(submission, best['test_proba'], le, name='eunji_lr_l0_C0.1', cv_score=best['cv_mean'])
pd.Series(le.classes_[best['test_proba'].argmax(axis=1)]).value_counts(normalize=True).round(3).head(8)

[fold 0] Macro F1 = 0.3872
[fold 1] Macro F1 = 0.3733
[fold 2] Macro F1 = 0.3831
[fold 3] Macro F1 = 0.3755
[fold 4] Macro F1 = 0.3898
CV Macro F1 = 0.3818 ± 0.0064 (OOF 0.3830)
저장 완료: /Users/mac/web/dacon-cancer-subclass/submissions/1109_eunji_lr_l0_C0.1_0.382.csv


COAD    0.225
STES    0.079
THYM    0.066
OV      0.065
BRCA    0.062
PCPG    0.045
TGCT    0.036
LGG     0.035
Name: proportion, dtype: float64